In [1]:
import json
import unidecode
import contractions
from nltk import sent_tokenize, word_tokenize
import nltk
import logging
import re
import string
from sklearn.metrics import confusion_matrix
from seqeval.metrics import classification_report, accuracy_score, f1_score, recall_score, precision_score
from seqeval.scheme import IOB2

logger = logging.getLogger(__name__)
stop_words = nltk.corpus.stopwords.words('english')
#deselect 'no' and 'not' from stop words
stop_words.remove('no')
stop_words.remove('not')

## Do not remove '.' ',' ';' ':'
english_punctuations = ['``','?', '（','）','(', ')',
                '[', ']', '&', '!', '*', '@', '#', '$', '%','\\','\"','}','{','=','/','>','<','|','+','_','~']
english_punctuations_arr= ''.join(english_punctuations)

In [3]:
def read_json_array(path):
    '''Read JSON given array of dictionaries'''
    with open(path, "r", encoding="utf-8") as fp:
        datas = json.loads(fp.read())
    return datas

trainData = read_json_array("../data/pubmed_data/json_data_synthetic_labels/v2_train.json")
testData = read_json_array("../data/pubmed_data/json_data_synthetic_labels/v2_test.json")
gs42Data = read_json_array("../data/pubmed_data/json_data_synthetic_labels/v2_testgs42.json")
gs91Data = read_json_array("../data/pubmed_data/json_data_synthetic_labels/v2_testgs91.json")

In [4]:
def sent2words(sent):
    '''Pre-process sentence text to words'''
    
    ## Expand Words
    sent = contractions.fix(sent)
    
    ## Convert sentence to lowercase and strip whitespaces
    sent = sent.lower().strip()
    
    ## Convert unicode characters in sentence to ASCII
    sent = unidecode.unidecode(sent)
    sent = sent.encode("ascii", "ignore").decode()
    
    ## Remove URL's from Sentence
    sent = re.sub('http[s]?://\S+', '', sent)
    
    ## Remove words like 2.0-2
    sent = re.sub(' \d*\.*\d*\-+\d*\.*\d* ', ' ', sent)
    
    # Remove punctuation characters
    sent = sent.translate(str.maketrans(english_punctuations_arr,' '*len(english_punctuations_arr)))
    
    pos, length = {}, {}
    words = word_tokenize(sent)
    for i, (w, p) in enumerate(nltk.pos_tag(words)):
        # Computational features
        pos[w] = p
        length[w] = len(w)
    
    # Remove Stop Words
    # words = [word for word in word_tokenize(sent) if not word in stop_words]
    
    # Remove punctuation only strings from sentence
    words = [word for word in words if not all(c in string.punctuation for c in word) and len(word)>1]
    
    # Remove Numericals from sentence
    words = [x for x in words if not all(c.isdigit() for c in x)] 
    
    ## Return words, POS tags, lengths of words, processed sentence
    return words, pos, length, ' '.join(words)

# Mark Keyphrases with BIO Format Labels
def mark_keyword_all_sentences(keywords, sentences):

    sentence_lengths = [len(sent) for sent in sentences]
    logger.debug("Sentence length: %s"%sentence_lengths)

    complete_text = []
    for sent in sentences:
        complete_text.extend(sent)
    logger.debug("Complete Text: %s"%complete_text)

    complete_text_len = len(complete_text)
    mapper = ['O']*complete_text_len
    kws = [sent2words(x)[0] for x in keywords]

    for kw in kws:
        kw_len=len(kw)
        if kw_len == 0:
            continue
        i=0
        while i<complete_text_len-kw_len:
            if complete_text[i:i+kw_len]==kw and mapper[i:i+kw_len]==['O']*kw_len:
                mapper[i:i+kw_len]=['I-KP']*kw_len
                mapper[i]='B-KP'
            i+=1

    final_mapper = []
    final_tokens = []
    start=0

    for slen in sentence_lengths:
        final_mapper.append(mapper[start:start+slen])
        final_tokens.append(complete_text[start:start+slen])
        start+=slen
    
    return complete_text, final_mapper, final_tokens

In [5]:
def createBERTtSV(data,filename):
    # import csv
    # trainCSV = "../data/trainBERT.tsv"
    f = open(filename,'w')
    f.close()

    for article in data:
        textData = []
        for sent in sent_tokenize(article['title']+' '+article['abstract']):
            words, _, _, _ = sent2words(sent)
            # words.append('.')
            textData.append(words)
        # textData = [word_tokenize(sent) for sent in sent_tokenize(article['title']+' '+article['abstract'])]
        #print(textData)
        _, mapper, tokens = mark_keyword_all_sentences(article['keywords'],textData)
        with open(filename,'a') as f:
        # with open(trainCSV, 'w', encoding='utf8', newline='') as tsv_file:
        #     tsv_writer = csv.writer(tsv_file, delimiter='\t', lineterminator='\n')
        #     tsv_writer.writerow(["Word", "Count"])
            print("-DOCSTART- -X- O O", file=f)
            for sentM, sentT in zip(mapper, tokens):
                for wordM, wordT in zip(sentM, sentT):
                    print("{}\t{}".format(wordT,wordM), file=f)
        # print(mapper, tokens)
        
def getTargetPreds(data):
    comb_arr = []
    for article in data:
        comb_arr.append(article['title']+' '+article['abstract'])

    y_preds, y_targets = [], []
    for idx, doc in enumerate(nlp.pipe(comb_arr)):

        kws = [str(x) for x in doc.ents]
        textData = []
        for sent in sent_tokenize(article['title']+' '+article['abstract']):
            words, _, _, _ = sent2words(sent)
            textData.append(words)
        # textData = [word_tokenize(sent) for sent in sent_tokenize(data[idx]['title']+' '+data[idx]['abstract'])]

        _, target, _ = mark_keyword_all_sentences(data[idx]['keywords'],textData)
        _, predict, _ = mark_keyword_all_sentences(kws,textData)
        #print(gs40Data[idx]['id'],gs40Data[idx]['keywords'],kws,sep='\n')

        #print(target,predict,sep='\n')
        y_preds.extend(predict)
        y_targets.extend(target)
    
    return y_preds, y_targets

In [7]:
totalData = trainData+testData
datalen = len(totalData)
print("Data: {}, Train: {} Val: {} Test: {}".format(datalen, datalen*0.6, datalen*0.2, datalen*0.2))
createBERTtSV(totalData[:int(0.6*datalen)],'../data/trainSpacy.tsv')
createBERTtSV(totalData[int(0.6*datalen):int(0.8*datalen)],'../data/validSpacy.tsv')
createBERTtSV(totalData[int(0.8*datalen):],'../data/testSpacy.tsv')

createBERTtSV(gs42Data,'../data/gs42SpacyFull.tsv')
gs_datalen = len(gs42Data)
print("GS42 Data: {}, GS42 Train: {} GS42 Val: {}".format(gs_datalen, gs_datalen*0.8, gs_datalen*0.2))
createBERTtSV(gs42Data[:int(0.8*gs_datalen)],'../data/gs42Spacy_T.tsv')
createBERTtSV(gs42Data[int(0.8*gs_datalen):],'../data/gs42Spacy_V.tsv')

createBERTtSV(gs91Data,'../data/gs91SpacyFull.tsv')
gs_datalen = len(gs91Data)
print("GS91 Data: {}, GS91 Train: {} GS91 Val: {}".format(gs_datalen, gs_datalen*0.8, gs_datalen*0.2))
createBERTtSV(gs91Data[:int(0.8*gs_datalen)],'../data/gs91Spacy_T.tsv')
createBERTtSV(gs91Data[int(0.8*gs_datalen):],'../data/gs91Spacy_V.tsv')

Data: 3110, Train: 1866.0 Val: 622.0 Test: 622.0
GS42 Data: 42, GS42 Train: 33.6 GS42 Val: 8.4
GS91 Data: 91, GS91 Train: 72.8 GS91 Val: 18.2


In [11]:
import sys
!cd ../data && {sys.executable} -m spacy convert trainSpacy.tsv ./ -t json -n 1 -c iob
!cd ../data && {sys.executable} -m spacy convert validSpacy.tsv ./ -t json -n 1 -c iob
!cd ../data && {sys.executable} -m spacy convert testSpacy.tsv ./ -t json -n 1 -c iob
!cd ../data && {sys.executable} -m spacy convert gs42SpacyFull.tsv ./ -t json -n 1 -c iob
!cd ../data && {sys.executable} -m spacy convert gs42Spacy_T.tsv ./ -t json -n 1 -c iob
!cd ../data && {sys.executable} -m spacy convert gs42Spacy_V.tsv ./ -t json -n 1 -c iob
!cd ../data && {sys.executable} -m spacy convert gs91SpacyFull.tsv ./ -t json -n 1 -c iob
!cd ../data && {sys.executable} -m spacy convert gs91Spacy_T.tsv ./ -t json -n 1 -c iob
!cd ../data && {sys.executable} -m spacy convert gs91Spacy_V.tsv ./ -t json -n 1 -c iob

ℹ Auto-detected token-per-line NER format
⚠ Document delimiters found, automatic document segmentation with `-n`
disabled.
⚠ No sentence boundaries found. Use `-s` to automatically segment
sentences.
✔ Generated output file (1 documents): trainSpacy.json
ℹ Auto-detected token-per-line NER format
⚠ Document delimiters found, automatic document segmentation with `-n`
disabled.
⚠ No sentence boundaries found. Use `-s` to automatically segment
sentences.
✔ Generated output file (1 documents): validSpacy.json
ℹ Auto-detected token-per-line NER format
⚠ Document delimiters found, automatic document segmentation with `-n`
disabled.
⚠ No sentence boundaries found. Use `-s` to automatically segment
sentences.
✔ Generated output file (1 documents): testSpacy.json
ℹ Auto-detected token-per-line NER format
⚠ Document delimiters found, automatic document segmentation with `-n`
disabled.
⚠ No sentence boundaries found. Use `-s` to automatically segment
sentences.
✔ Generated output file (1 documents

In [12]:
!cd ../data && {sys.executable} -m spacy convert trainSpacy.json ./ -t spacy
!cd ../data && {sys.executable} -m spacy convert validSpacy.json ./ -t spacy
!cd ../data && {sys.executable} -m spacy convert testSpacy.json ./ -t spacy
!cd ../data && {sys.executable} -m spacy convert gs42SpacyFull.json ./ -t spacy
!cd ../data && {sys.executable} -m spacy convert gs42Spacy_T.json ./ -t spacy
!cd ../data && {sys.executable} -m spacy convert gs42Spacy_V.json ./ -t spacy
!cd ../data && {sys.executable} -m spacy convert gs91SpacyFull.json ./ -t spacy
!cd ../data && {sys.executable} -m spacy convert gs91Spacy_T.json ./ -t spacy
!cd ../data && {sys.executable} -m spacy convert gs91Spacy_V.json ./ -t spacy

✔ Generated output file (1866 documents): trainSpacy.spacy
✔ Generated output file (622 documents): validSpacy.spacy
✔ Generated output file (622 documents): testSpacy.spacy
✔ Generated output file (42 documents): gs42SpacyFull.spacy
✔ Generated output file (33 documents): gs42Spacy_T.spacy
✔ Generated output file (9 documents): gs42Spacy_V.spacy
✔ Generated output file (91 documents): gs91SpacyFull.spacy
✔ Generated output file (72 documents): gs91Spacy_T.spacy
✔ Generated output file (19 documents): gs91Spacy_V.spacy


### Round 1:
#### Stage 1: Fine-Tune Bio-Medical sciSpacy to CDSS Domain [Common to BOTH Experiments]
sciSpacy >> Train/Val Synthetic DS >> cdssSciSpacy << Test Synthetic DS
#### Stage 2: Fine-Tune Synthetic-CDSS-sciSpacy to GS-CDSS-sciSpacy
cdssSciSpacy >> Train/Val GS42 DS >> cdssSciSpacyGS << Test G291 DS

### Round 2:
#### Stage 1: Fine-Tune Bio-Medical sciSpacy to CDSS Domain [Common to BOTH Experiments]
sciSpacy >> Train/Val Synthetic DS >> cdssSciSpacy << Test Synthetic DS
#### Stage 2: Fine-Tune Synthetic-CDSS-sciSpacy to GS-CDSS-sciSpacy
cdssSciSpacy >> Train/Val GS91 DS >> cdssSciSpacyGS << Test G242 DS